---
title: "4.5 Fixed effects and panel models"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/04-regression/05-fixed-effects.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/04-regression/05-fixed-effects.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [4.4 Robust and clustered standard errors](../04-regression/04-robust-clustered.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Describe which variation country and year fixed effects use.
- Fit a two-way fixed-effects model on a country-year index.
- Recognize absorbed variables and limits on causal interpretation.

## Why it matters

A cross-country comparison mixes persistent country differences with changes over time. **Country fixed effects** allow each country its own constant fitted level, so the income slope uses within-country variation rather than fixed differences between countries. **Year fixed effects** allow a separate shared fitted level for each observed year. This changes the comparison; it does not eliminate all time-varying confounding, reverse causation or measurement error. We start with made-up numbers and then fit the historical panel.

## Worked example

### Prepare the analysis libraries

Run this prepared cell **before importing other analysis libraries**. It uses the versions generated from the course's single dependency file. If needed, it installs free packages inside this temporary Python runtime, not on your own computer. An internet connection is needed for installation. If it reports that a restart is required, restart the Colab session and run from the top; mixing already loaded old code with newly installed libraries can produce misleading errors.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared analysis-library setup (run first)"
# Generated from requirements.txt; installs only in this Python runtime.
import importlib.metadata
from pathlib import Path
import subprocess
import sys

requested = ['statsmodels==0.15.0', 'linearmodels==7.1']
needs_install = False
for requirement in requested:
    name, version = requirement.split("==")
    try:
        needs_install = needs_install or importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ["numpy", "pandas", "scipy", "statsmodels", "linearmodels"])
    constraints = Path("scratch/course-analysis-constraints.txt")
    constraints.parent.mkdir(parents=True, exist_ok=True)
    constraints.write_text('# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n', encoding='utf-8')
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", str(constraints), *requested], check=True)
    if loaded:
        raise RuntimeError("Packages installed. Restart the notebook session, then run all cells from the top so updated libraries are used together.")
print("Analysis libraries ready:", ", ".join(requested))

Look for **Analysis libraries ready**. In the pinned course environment this cell installs nothing. Colab may take a few minutes the first time. The current course environment uses Python 3.12; an older runtime that cannot install the pinned dependencies needs a current Python runtime rather than arbitrary version substitutions.

### Subtract a country's own mean

The small table contains made-up numbers for two fictional countries over three periods. `.transform("mean")` returns each group's mean aligned with every original row. Subtracting it removes a country-specific constant level. This is the idea of a **within transformation**; it does not yet remove shared time changes.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
small = pd.DataFrame({
    "country": ["Northland"] * 3 + ["Eastria"] * 3,
    "year": [2000, 2005, 2010] * 2,
    "gdp_thousands": [8.0, 10.0, 12.0, 20.0, 22.0, 24.0],
    "life_exp": [60.0, 63.0, 64.0, 70.0, 71.0, 74.0],
})
small["gdp_within"] = small["gdp_thousands"] - small.groupby("country")["gdp_thousands"].transform("mean")
small["life_within"] = small["life_exp"] - small.groupby("country")["life_exp"].transform("mean")
small[["country", "year", "gdp_within", "life_within"]]

The deviations average to zero within each country. Northland and Eastria have different original levels but the same pattern of GDP deviations here. These six invented observations cannot justify country-clustered inference from only two clusters.

### Fit the toy country fixed-effects model

A **MultiIndex** is a row label made from multiple fields. PanelOLS expects the first level to identify the unit and the second to identify time. In its formula, `EntityEffects` means unit (here country) fixed effects. Write `1` explicitly for the intercept; linearmodels formula conventions differ from the automatic-intercept convention used by statsmodels. The toy uses unadjusted covariance only to demonstrate a coefficient, not to make an uncertainty claim.

In [ ]:
toy_panel = small.set_index(["country", "year"]).sort_index()
toy_fe = PanelOLS.from_formula("life_exp ~ 1 + gdp_thousands + EntityEffects", data=toy_panel).fit(cov_type="unadjusted")
toy_dummy = smf.ols("life_exp ~ gdp_thousands + C(country)", data=small).fit()
print("PanelOLS slope:", toy_fe.params["gdp_thousands"])
print("Country-indicator OLS slope:", toy_dummy.params["gdp_thousands"])

The two slopes agree at about one year per additional thousand made-up dollars. Explicit country indicator columns and an equivalent fixed-effects transformation can estimate the same slope. The representation and uncertainty defaults may differ, so agreement on a slope does not establish agreement on standard errors.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Check the real panel keys

Load complete observations, verify positive GDP and create log income. Check country-year uniqueness before making the two-level index. Duplicate country-years should be resolved substantively rather than treated as extra repeated measurements by accident.

In [ ]:
countries = pd.read_csv(data_file)
analysis = countries[["country", "region", "year", "life_exp", "gdp_pc"]].dropna().copy()
print("Repeated country-years:", analysis.duplicated(["country", "year"]).sum())
print("Positive GDP:", analysis["gdp_pc"].gt(0).all())
analysis["log_gdp_pc"] = np.log(analysis["gdp_pc"])
panel = analysis.set_index(["country", "year"]).sort_index()
print("Country-years:", len(panel))
print("Countries:", panel.index.get_level_values("country").nunique())
print("Observed years:", panel.index.get_level_values("year").nunique())

The keys are unique, with 1704 country-years, 142 countries and 12 observed years. This filtered panel is balanced: all countries have the same twelve five-year observations. Balanced data do not establish that countries were sampled representatively.

### Add country and year effects

First estimate country effects only; then add `TimeEffects` for shared year-level differences. Both fits use country-clustered covariance with a finite-sample adjustment. The clusters are the indexed entities, selected by `cluster_entity=True`, so their order stays aligned with the model data.

In [ ]:
country_fe = PanelOLS.from_formula("life_exp ~ 1 + log_gdp_pc + EntityEffects", data=panel).fit(cov_type="clustered", cluster_entity=True, debiased=True)
two_way = PanelOLS.from_formula("life_exp ~ 1 + log_gdp_pc + EntityEffects + TimeEffects", data=panel).fit(cov_type="clustered", cluster_entity=True, debiased=True)
fe_comparison = pd.DataFrame({
    "country_FE_only": [country_fe.params["log_gdp_pc"], country_fe.std_errors["log_gdp_pc"]],
    "country_and_year_FE": [two_way.params["log_gdp_pc"], two_way.std_errors["log_gdp_pc"]],
}, index=["Income coefficient (years per log unit)", "Country-clustered standard error"])
fe_comparison.round(4)

With both country and year effects, the income coefficient is about 1.450 years per log unit and its country-clustered standard error about 0.707. The year effects remove shared fitted time changes; they do not remove every country-specific time-varying factor or cross-country correlated error. This slope is not the same estimand (the quantity being estimated) as a single-year cross-country slope. Their differences do not tell you which one is automatically causal.

### Interpret a within-country ratio comparison

Multiply the two-way slope by log(1.10) to describe 10% higher GDP per person in the fitted within-country, year-adjusted comparison. Report an interval from the model's own covariance implementation rather than borrowing a cross-section's uncertainty.

In [ ]:
fe_ten_percent = two_way.params["log_gdp_pc"] * np.log(1.10)
print("Fitted 10% comparison, years:", round(fe_ten_percent, 3))
print(two_way.conf_int().loc[["log_gdp_pc"]].round(4))

The fitted comparison is about 0.138 years. A causal interpretation would require additional assumptions: for example, no unobserved time-varying determinants linked to income, and appropriate treatment of reverse causation and dynamics. Fixed effects also cannot separately identify the coefficient of a predictor that never varies within a country once country effects absorb it. Model fit measures reported by different panel packages need not use the same definition as ordinary cross-section R-squared.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Check the fixed-effect settings

Print whether the two-way model includes entity and time effects and report its observation count.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
print(two_way.model.entity_effects)
print(two_way.model.time_effects)
print(int(two_way.nobs))

The settings are True, True and 1704. Confirm the fitted model actually matches the formula you intended before interpreting a result.

</details>

### 2. Explain a fixed attribute

Count distinct region labels within each country. Can the model distinguish a separate constant-region effect from the country fixed effects?

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
regions_per_country = analysis.groupby("country")["region"].nunique()
print(regions_per_country.value_counts())

All 142 countries have one region label in this extract. A constant region indicator is a combination of country indicators, so it adds no separately identifiable variation after country effects.

</details>

### 3. Compare point estimates in the toy

Use np.isclose to compare the toy PanelOLS and country-indicator OLS slopes. Explain what this check does and does not establish.

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
toy_matches = np.isclose(toy_fe.params["gdp_thousands"], toy_dummy.params["gdp_thousands"])
print(toy_matches)

The result is True. It checks an algebraic representation on the toy data. It does not validate sampling assumptions, clustered inference from two countries, or the real-world causal interpretation of either model.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### A time-invariant category is absorbed

The package's AbsorbingEffectError identifies predictors explained entirely by the included effects.

In [ ]:
try:
    from linearmodels.panel.utility import AbsorbingEffectError
    PanelOLS.from_formula("life_exp ~ 1 + log_gdp_pc + C(region) + EntityEffects + TimeEffects", data=panel).fit()
except AbsorbingEffectError as error:
    print(f"{type(error).__name__}: {error}")

Country effects already span the region indicators. Remove an unidentifiable term only after explaining the comparison it cannot estimate; turning off checks or dropping terms blindly is not a research strategy.

### A duplicate observation needs an explicit guard

Some estimators may accept repeated index labels as additional observations. This deliberately adds one duplicate and uses our own check to reject it.

In [ ]:
try:
    duplicated_panel = pd.concat([panel, panel.iloc[[0]]])
    if duplicated_panel.index.duplicated().any():
        raise ValueError("Duplicate country-year keys: review the observation definition before estimation.")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Inspect the duplicate source records and intended unit of observation. Do not keep both, average them, or drop one merely to make the key check pass without a substantive reason.

## Recap

- Country effects use within-country variation; year effects remove shared fitted year levels.
- Index unit and time correctly and verify unique keys.
- Country-clustered covariance still needs appropriate dependence assumptions.
- Absorbed variables and remaining time-varying confounding limit what can be learned.

**Next:** [4.6 Regression tables you can share](../04-regression/06-regression-tables.ipynb).

::: {.callout-note title="Coming from Stata / R / Excel?"}
EntityEffects and TimeEffects play the role of country and year fixed effects in a panel regression. Check package-specific intercept, covariance and degrees-of-freedom conventions before comparing output from Stata, R or another Python package.
:::